# Retrosynthesis Green Chemistry Evaluator

Evaluates retrosynthetic routes against the 12 Principles of Green Chemistry using PubChem hazard data and an LLM scoring engine.

**Setup:** Set your Azure OpenAI credentials as environment variables before running:
```bash
export AZURE_OPENAI_KEY='your-key-here'
export AZURE_OPENAI_ENDPOINT='https://your-endpoint.azure-api.net'
```

In [1]:
import re
import json
import os
import time
import urllib.parse
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

In [2]:
# ---------------------------------------------------------------------------
# PubChem helpers
# ---------------------------------------------------------------------------

HEADERS = {
    "User-Agent": "Mozilla/5.0 (retrosynthesis-green-eval/1.0; contact: you@example.com)"
}

H_CODE_RE = re.compile(r"\bH\d{3}\b")

MAX_WORKERS = 5          # parallel PubChem threads
REQUEST_DELAY = 0.2      # seconds between requests per thread
MAX_RETRIES = 3          # retry attempts on transient failures


def _get_with_retry(url, retries=MAX_RETRIES):
    """GET with simple exponential back-off retry on network/SSL errors."""
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            return resp
        except Exception as exc:
            wait = 2 ** attempt
            if attempt < retries - 1:
                time.sleep(wait)
            else:
                raise exc


def get_cid_from_smiles(smiles):
    """Resolve a SMILES string to a PubChem CID via PUG REST."""
    encoded = urllib.parse.quote(smiles, safe="")
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/{encoded}/cids/JSON"
    try:
        resp = _get_with_retry(url)
        if resp.status_code == 200:
            cids = resp.json().get("IdentifierList", {}).get("CID", [])
            if cids:
                return cids[0]
    except Exception as exc:
        print(f"[WARN] CID lookup failed for {smiles[:40]}…: {exc}")
    return None


def _collect_hcodes_recursive(obj, out_set):
    """Recursively scan a PubChem PUG View JSON object and collect all H-codes."""
    if isinstance(obj, dict):
        for v in obj.values():
            _collect_hcodes_recursive(v, out_set)
    elif isinstance(obj, list):
        for item in obj:
            _collect_hcodes_recursive(item, out_set)
    elif isinstance(obj, str):
        for m in H_CODE_RE.findall(obj):
            out_set.add(m)


def get_ghs_hazards(cid):
    """Return sorted list of GHS H-codes for a given PubChem CID."""
    url = (
        f"https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound/{cid}/JSON/"
        "?heading=GHS+Classification"
    )
    h_codes: set = set()
    try:
        resp = _get_with_retry(url)
        if resp.status_code == 200:
            _collect_hcodes_recursive(resp.json(), h_codes)
    except Exception as exc:
        print(f"[WARN] GHS lookup failed for CID {cid}: {exc}")
    return sorted(h_codes)


def _lookup_smiles(smiles, cid_cache, ghs_cache):
    """Thread-safe worker: resolve SMILES → CID → H-codes."""
    if len(smiles) < 3:
        return smiles, []

    cid = cid_cache.get(smiles)
    if cid is None:
        cid = get_cid_from_smiles(smiles)
        cid_cache[smiles] = cid  # may be None

    if not cid:
        return smiles, []

    if cid in ghs_cache:
        return smiles, ghs_cache[cid]

    time.sleep(REQUEST_DELAY)
    h_codes = get_ghs_hazards(cid)
    ghs_cache[cid] = h_codes
    return smiles, h_codes

In [3]:
# ---------------------------------------------------------------------------
# SMILES extraction
# ---------------------------------------------------------------------------

def extract_unique_smiles(graph_text):
    """
    Parse all unique SMILES from retrosynthesis graph text.

    Handles:
      Product: <SMILES>
      Reactants: [SMILES, SMILES, ...]

    SMILES containing brackets (e.g. [O-]) are captured correctly by
    matching to end-of-line rather than the first ']'.
    """
    smiles_set: set = set()

    # Product lines
    for m in re.finditer(r"^\s*Product:\s*(\S+)\s*$", graph_text, flags=re.M):
        smiles_set.add(m.group(1).strip())

    # Reactant lines  — greedy match to last ']' on the line
    for m in re.finditer(r"^\s*Reactants:\s*\[(.*)\]\s*$", graph_text, flags=re.M):
        content = m.group(1).strip()
        parts = [p.strip().strip("'\"") for p in content.split(",")]
        smiles_set.update(smi for smi in parts if smi)

    return smiles_set

In [4]:
# ---------------------------------------------------------------------------
# Prompt construction
# ---------------------------------------------------------------------------

SYSTEM_PROMPT = """
You are an elite expert in cheminformatics, green chemistry, and process engineering.

Your objective is to systematically evaluate multiple computational retrosynthetic routes
("Graphs") according to the 12 Principles of Green Chemistry, using deterministic
rule-based scoring.

You will receive:
  1) A plain-text string containing multiple retrosynthesis routes labeled
     "=== Graph 1 ===", "=== Graph 2 ===", etc.
  2) An "External Hazard Data" dictionary mapping SMILES to GHS H-codes.

Each graph uses this step format:
  Step N: <reaction type>
    Reactants: [SMILES, ...]
    Product: SMILES
    Atom economy: XX.XX%
    PMI: X.XX

You MUST process ALL graphs in the input. Do not truncate.

------------------------------------------------------------
I. REQUIRED PARSING (for each Graph)

For each graph extract:
  graph_id               – integer after "Graph "
  atom_economy_percent   – float from "Atom economy: XX.XX%"
  total_steps            – count of "Step n:" entries
  pmi_values             – list of PMI floats (skip steps that lack PMI)
  route_mean_PMI         – arithmetic mean of pmi_values
                           (0.0 if no PMI values found)

DATA VALIDATION:
  • atom_economy_percent > 100.0 is a physically impossible value and indicates
    a parsing anomaly. Flag it (add "AE_anomaly" to hazard_flags_detected) and
    cap AE_score at 1.0 for scoring: AE_score = min(atom_economy_percent / 100.0, 1.0)

------------------------------------------------------------
II. GREEN CHEMISTRY SCORING COMPONENTS

1) Mass Efficiency & Catalysis  (Principles 1, 2, 9)

  AE_score = min(atom_economy_percent / 100.0, 1.0)   ← cap at 1.0

  WP_score = 1.0 / (1.0 + route_mean_PMI)

  Catalytic classification — a step is catalytic when ANY of the following apply:
    a) The reaction type name matches a known catalytic transformation:
       Suzuki, Heck, Sonogashira, Buchwald–Hartwig, Ullmann,
       hydrogenation, transfer hydrogenation, metal-catalyzed
       oxidation/reduction, or other well-known named catalytic reactions.
    b) A catalyst SMILES is present in the reactant list
       (e.g. Pd/C, PdCl2, Pd(PPh3)4, RhCl3, RuCl3, Ni complexes, etc.).
    c) A known organocatalyst or enzyme is listed (DMAP, proline, lipase, etc.).
    Mark non-catalytic ONLY if none of (a)–(c) apply.

  CAT_score = catalytic_steps / total_steps
              (0.0 if total_steps = 0)

------------------------------------------------------------
2) Reduce Derivatives  (Principle 8)

  Scan intermediate SMILES for temporary protecting groups:
    Boc, Fmoc, Cbz, THP, TBDMS, MOM, Trityl.

  NOTE: Benzyl ether (Bn) groups are counted here as protecting groups,
  NOT as hazard flags. Do NOT include "Benzyl ether" in hazard_flags_detected.

  derivatization_cycles = number of complete protection–deprotection cycles detected
  DP_pen = 0.15 × derivatization_cycles

------------------------------------------------------------
3) Renewable Feedstocks  (Principle 7)

  Examine terminal leaf-node (starting material) SMILES.
  Check whether any is a primary derivative of the DOE Top-12 bio-based platform
  chemicals: succinic acid, lactic acid, glycerol, furfural, levulinic acid,
  itaconic acid, sorbitol.

  uses_renewable_feedstock = true if detected, else false
  REN_score = 0.10 if true, else 0.0

------------------------------------------------------------
4) Hazard & Accident Prevention  (Principles 3, 4, 12)

  A) Structural SMARTS scan of all reactant and product SMILES:
     Explosive/reactive:
       Azides:              [NX1]-[NX2+]=[NX1-]
       Peroxides:           [OX2]-[OX2]
       Poly-nitro aromatics
     Highly toxic/corrosive:
       Phosgene:            ClC(=O)Cl
       Methyl iodide:       CI
       Cyanides:            C#N
       Acid halides:        C(=O)[F,Cl,Br,I]

  B) External Hazard Data check — if any SMILES maps to severe GHS codes:
       Explosive:           H200–H211
       Fatal acute toxicity: H300, H310, H330

  HAZ_pen = 0.20 if severe hazards detected, else 0.0
  Record all detected hazard type names in hazard_flags_detected (string array).
  Do NOT include "Benzyl ether" here — see Section II-2.

------------------------------------------------------------
5) Solvents & Energy Efficiency  (Principles 5 & 6)

  If solvent or temperature data is present in the graph, apply:
    Energy-intensive: cryogenic (−78 °C organolithium) or prolonged high-T reflux.
    Problematic solvents: DCM, DMF, diethyl ether.
    Biocatalysis / aqueous enzymatic steps count as a bonus.

    ES_pen = 0.10   (energy-intensive or toxic-solvent dependent)
    ES_pen = −0.05  (aqueous biocatalysis confirmed)
    ES_pen = 0.0    (insufficient data — default)

  If no solvent or temperature data is available, set ES_pen = 0.0 and add
  "ES_data_unavailable" to hazard_flags_detected for transparency.

------------------------------------------------------------
6) Design for Degradation  (Principle 10)

  If the final product or major byproduct contains extensive polyfluorination
  (e.g. C(F)(F)F groups) with no degradable handles:
    DEG_pen = 0.05
  Else:
    DEG_pen = 0.0

------------------------------------------------------------
III. COMBINED GREEN SCORE

Compute after all components are finalized:

  overall_score =
      (0.25 × AE_score)
    + (0.25 × WP_score)
    + (0.15 × CAT_score)
    + REN_score
    − DP_pen
    − HAZ_pen
    − ES_pen
    − DEG_pen

  Clamp: overall_score = max(0.0, min(1.0, overall_score))

------------------------------------------------------------
IV. OUTPUT FORMAT (STRICT)

Output ONLY a valid JSON array. No markdown, no explanations,
no extra text, no trailing commas.
Do NOT include a "rank" field — ranking is computed in post-processing.

Each element MUST follow exactly:
{
  "graph_id": <int>,
  "atom_economy_percent": <float>,
  "catalytic_step_ratio": <float>,
  "route_mean_PMI": <float>,
  "derivatization_cycles": <int>,
  "uses_renewable_feedstock": <boolean>,
  "hazard_flags_detected": [<string>, ...],
  "overall_score": <float>
}

Process ALL graphs. Return strictly valid JSON only.
"""


def build_user_prompt(graph_text, hazard_json_str):
    return f"""Below is hazard context retrieved from PubChem, followed by the retrosynthesis graphs.
Each graph is labelled "=== Graph N ===" and contains steps in this format:
  Step N: <reaction type>
    Reactants: [SMILES, ...]
    Product: SMILES
    Atom economy: XX.XX%
    PMI: X.XX

External Hazard Data (GHS H-Codes from PubChem):
{hazard_json_str}

Retrosynthesis Graphs:
{graph_text}
"""

In [5]:
# ---------------------------------------------------------------------------
# Main evaluation function
# ---------------------------------------------------------------------------

def _assign_ranks(results):
    """
    Assign ranks in Python based on overall_score (descending).
    Uses dense ranking: tied scores share the same rank,
    next rank is consecutive with no gaps.
    Replaces the LLM-generated rank which is unreliable.
    """
    sorted_results = sorted(results, key=lambda x: x["overall_score"], reverse=True)
    rank = 1
    for i, entry in enumerate(sorted_results):
        if i > 0 and entry["overall_score"] < sorted_results[i - 1]["overall_score"]:
            rank = i + 1
        entry["rank"] = rank
    return sorted_results


def evaluate_retrosynthesis_routes(graph_text, llm_client):
    """
    Full pipeline:
      1. Extract unique SMILES from graph_text.
      2. Query PubChem for GHS hazard H-codes (parallel, with retry).
      3. Build prompt and call the LLM.
      4. Parse, validate, re-rank in Python, and return the result list.
    """
    # ------------------------------------------------------------------
    # Step 1 — SMILES extraction
    # ------------------------------------------------------------------
    print("[1/3] Extracting SMILES and querying PubChem…")
    unique_smiles = extract_unique_smiles(graph_text)
    print(f"  Unique SMILES found: {len(unique_smiles)}")
    print(f"  Sample (up to 10): {list(unique_smiles)[:10]}")

    # ------------------------------------------------------------------
    # Step 2 — Parallel PubChem lookups
    # ------------------------------------------------------------------
    hazard_context = {}
    cid_cache: dict = {}
    ghs_cache: dict = {}

    smiles_to_lookup = [s for s in unique_smiles if len(s) >= 3]

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(_lookup_smiles, smi, cid_cache, ghs_cache): smi
            for smi in smiles_to_lookup
        }
        for future in tqdm(as_completed(futures), total=len(futures),
                           desc="  PubChem lookup"):
            smi, h_codes = future.result()
            if h_codes:
                hazard_context[smi] = h_codes

    hazard_json_str = json.dumps(hazard_context, indent=2, ensure_ascii=False)
    print(f"  Hazard entries retrieved: {len(hazard_context)}")

    # ------------------------------------------------------------------
    # Step 3 — LLM call
    # Rank is intentionally excluded from the LLM output spec;
    # Python assigns it correctly in Step 4.
    # ------------------------------------------------------------------
    print("[2/3] Calling LLM…")
    response = llm_client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_user_prompt(graph_text, hazard_json_str)},
        ],
        max_tokens=32768,
        temperature=0,
    )
    raw = response.choices[0].message.content

    # ------------------------------------------------------------------
    # Step 4 — Parse, validate, and re-rank
    # ------------------------------------------------------------------
    print("[3/3] Parsing, validating and ranking…")
    clean = re.sub(r"```(?:json)?\s*", "", raw).strip()
    try:
        results = json.loads(clean)
    except json.JSONDecodeError as exc:
        print(f"[ERROR] JSON parse failed: {exc}")
        print("Raw LLM output:\n", raw[:2000])
        return None

    # Ranks are assigned here in Python — never trust LLM-generated rank values
    results_ranked = _assign_ranks(results)

    # Quick summary
    print(f"\n✅ Evaluated {len(results_ranked)} routes.")
    top = results_ranked[0]
    print(
        f"   🥇 Best route: Graph {top['graph_id']}  "
        f"score={top['overall_score']:.3f}  "
        f"AE={top['atom_economy_percent']:.1f}%  "
        f"PMI={top['route_mean_PMI']:.2f}"
    )

    return results_ranked


In [7]:
# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------
# Credentials are read from environment variables — never hard-code API keys.

import openai
from openai import AzureOpenAI

llm_client = AzureOpenAI(
    api_key="5ba0576a77d140538e02d990919f8a14",
    api_version="2025-02-01-preview",
    azure_endpoint="https://hkust.azure-api.net",
)

# Path to your retrosynthesis graph text file
file_path = "./dataoutput/Aspirin.txt"

with open(file_path, "r", encoding="utf-8") as f:
    actual_graph_text = f.read()

results = evaluate_retrosynthesis_routes(actual_graph_text, llm_client)

[1/3] Extracting SMILES and querying PubChem…
  Unique SMILES found: 102
  Sample (up to 10): ['COC(C)=O', 'CC(C)(C)OC(=O)c1ccccc1[Si](C)(C)C', 'CC(=O)Oc1ccccc1C=O', 'O=C(OCc1ccccc1Br)c1ccccc1O', 'COC(=O)c1ccccc1OCc1ccccc1', 'CC(=O)Cl', 'CC(=O)Oc1ccccc1B1OC(C)(C)C(C)(C)O1', '[C]=O', 'O=C(O)c1ccccc1B(O)O', 'CC(=O)OC(=O)c1ccccc1OC(C)=O']


  PubChem lookup: 100%|██████████| 99/99 [00:51<00:00,  1.94it/s]


  Hazard entries retrieved: 53
[2/3] Calling LLM…
[3/3] Parsing, validating and ranking…

✅ Evaluated 177 routes.
   🥇 Best route: Graph 18  score=0.610  AE=91.4%  PMI=1.09


---
## Single-drug evaluation
Run the two cells below to evaluate and save results for one drug at a time.

In [ ]:
# ---------------------------------------------------------------------------
# Display results
# ---------------------------------------------------------------------------
if results:
    print(json.dumps(results, indent=2, ensure_ascii=False))

In [ ]:
# ---------------------------------------------------------------------------
# (Optional) Save results to JSON file
# ---------------------------------------------------------------------------
if results:
    output_path = "./dataoutput/Aspirin_green_scores.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"Results saved to {output_path}")

---
## Batch evaluation
Run the cells below to evaluate **all** `.txt` files in `dataoutput/` in one go.

In [8]:
# ---------------------------------------------------------------------------
# Batch processing — additional imports
# ---------------------------------------------------------------------------
from pathlib import Path
from tqdm.notebook import tqdm as tqdm_nb


In [9]:
# ---------------------------------------------------------------------------
# Batch processing function
# ---------------------------------------------------------------------------

def batch_evaluate(
    txt_dir: str = "./dataoutput",
    scores_dir: str = "./dataoutput",
    llm_client=None,
    skip_existing: bool = True,
):
    """
    Process every .txt retrosynthesis graph file in txt_dir,
    run green chemistry evaluation, and save a _green_scores.json
    for each drug into scores_dir.

    Parameters
    ----------
    txt_dir       : folder containing <Drug>.txt files
    scores_dir    : folder where <Drug>_green_scores.json will be written
    llm_client    : initialised AzureOpenAI (or OpenAI) client
    skip_existing : if True, skip drugs whose _green_scores.json already exists
    """
    txt_path    = Path(txt_dir)
    scores_path = Path(scores_dir)
    scores_path.mkdir(parents=True, exist_ok=True)

    txt_files = sorted(txt_path.glob("*.txt"))
    if not txt_files:
        print(f"No .txt files found in {txt_dir}/")
        return {}

    print(f"Found {len(txt_files)} .txt file(s) to evaluate.\n")

    all_results = {}   # drug_name -> list of scored dicts
    summary     = []

    for txt_file in txt_files:
        drug_name   = txt_file.stem                          # e.g. "Aspirin"
        output_json = scores_path / f"{drug_name}_green_scores.json"

        # ── Skip if already done ──────────────────────────────────────────
        if skip_existing and output_json.exists():
            print(f"[{drug_name}] Already exists — skipping.  ({output_json})")
            with open(output_json, encoding="utf-8") as f:
                all_results[drug_name] = json.load(f)
            summary.append({"drug": drug_name, "status": "skipped (cached)",
                             "routes": len(all_results[drug_name])})
            continue

        # ── Read graph text ───────────────────────────────────────────────
        print(f"[{drug_name}] Reading {txt_file.name} ...")
        with open(txt_file, encoding="utf-8") as f:
            graph_text = f.read()

        if not graph_text.strip():
            print(f"  [WARN] File is empty — skipping.")
            summary.append({"drug": drug_name, "status": "skipped (empty)", "routes": 0})
            continue

        # ── Evaluate ──────────────────────────────────────────────────────
        try:
            results = evaluate_retrosynthesis_routes(graph_text, llm_client)
        except Exception as exc:
            print(f"  [ERROR] {exc}")
            summary.append({"drug": drug_name, "status": f"ERROR: {exc}", "routes": 0})
            continue

        if results is None:
            summary.append({"drug": drug_name, "status": "ERROR: LLM parse failed", "routes": 0})
            continue

        # ── Save ──────────────────────────────────────────────────────────
        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

        all_results[drug_name] = results
        top = results[0]
        summary.append({
            "drug":   drug_name,
            "status": "done",
            "routes": len(results),
            "best_graph":  top["graph_id"],
            "best_score":  top["overall_score"],
        })
        print(f"  Saved {len(results)} routes -> {output_json}")
        print()

    # ── Summary table ─────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("BATCH EVALUATION SUMMARY")
    print("=" * 60)
    print(f"  {'Drug':<20} {'Status':<28} {'Routes':>6}  Best (graph / score)")
    print("-" * 60)
    for s in summary:
        best = ""
        if s["status"] == "done":
            best = f"Graph {s['best_graph']} / {s['best_score']:.3f}"
        print(f"  {s['drug']:<20} {s['status']:<28} {s['routes']:>6}  {best}")
    print("=" * 60)

    return all_results


In [10]:
# ---------------------------------------------------------------------------
# Run batch evaluation
# (Set skip_existing=False to re-evaluate all files from scratch)
# ---------------------------------------------------------------------------

all_results = batch_evaluate(
    txt_dir      = "./dataoutput",
    scores_dir   = "./dataoutput",
    llm_client   = llm_client,
    skip_existing= True,
)


Found 10 .txt file(s) to evaluate.

[Aspirin] Already exists — skipping.  (dataoutput/Aspirin_green_scores.json)
[Atorvastatin] Reading Atorvastatin.txt ...
[1/3] Extracting SMILES and querying PubChem…
  Unique SMILES found: 4
  Sample (up to 10): ['CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F)cc2)n1CC[C@@H]1C[C@H](CC(=O)O)OC(C)(C)O1', 'CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F)cc2)n1CC[C@@H](O)C[C@@H](O)CC(=O)OC(C)(C)C', 'CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F)cc2)n1CC[C@@H](O)C[C@@H](O)CC(=O)O', 'CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F)cc2)n1CC[C@@H]1C[C@H](CC(=O)OC(C)(C)C)OC(C)(C)O1']


  PubChem lookup: 100%|██████████| 4/4 [00:02<00:00,  1.66it/s]


  Hazard entries retrieved: 3
[2/3] Calling LLM…
[3/3] Parsing, validating and ranking…

✅ Evaluated 3 routes.
   🥇 Best route: Graph 1  score=0.497  AE=93.3%  PMI=1.07
  Saved 3 routes -> dataoutput/Atorvastatin_green_scores.json

[Benzocaine] Reading Benzocaine.txt ...
[1/3] Extracting SMILES and querying PubChem…
  Unique SMILES found: 71
  Sample (up to 10): ['CCOC(=O)c1ccc(NN)cc1', 'O=S(=O)(Cl)C(F)(F)F', 'O=C(O)c1ccc([N+](=O)[O-])cc1', 'CCN(N=O)C(N)=O', 'O=C([O-])[O-]', 'Nc1ccc(CO)cc1', 'CCBr', 'O=C(n1ccnc1)n1ccnc1', 'O', 'CCOC(=O)c1ccc(I)cc1']


  PubChem lookup: 100%|██████████| 68/68 [00:40<00:00,  1.67it/s]


  Hazard entries retrieved: 48
[2/3] Calling LLM…
[3/3] Parsing, validating and ranking…

✅ Evaluated 88 routes.
   🥇 Best route: Graph 39  score=0.580  AE=98.2%  PMI=1.02
  Saved 88 routes -> dataoutput/Benzocaine_green_scores.json

[Celecoxib] Reading Celecoxib.txt ...
[1/3] Extracting SMILES and querying PubChem…
  Unique SMILES found: 39
  Sample (up to 10): ['FC(F)(F)c1cc(-c2ccc(Br)cc2)[nH]n1', 'CC1(C)OB(c2cc(C(F)(F)F)n[nH]2)OC1(C)C', 'Cc1ccc(C(=O)CC(=O)C(F)(F)F)cc1', 'NS(=O)(=O)c1ccc(-n2nc(C(F)(F)F)cc2Cl)cc1', 'NS(=O)(=O)c1ccc(-n2nc(C(F)(F)F)cc2I)cc1', 'O=S(=O)(Cl)c1ccc(I)cc1', 'NS(=O)(=O)c1ccc(B(O)O)cc1', 'CC1(C)OB(c2ccc(Br)cc2)OC1(C)C', 'NS(=O)(=O)c1ccc(Br)cc1', 'CS(=O)(=O)c1ccc(I)cc1']


...

  Hazard entries retrieved: 52
[2/3] Calling LLM…
[3/3] Parsing, validating and ranking…

✅ Evaluated 20 routes.
   🥇 Best route: Graph 2  score=0.466  AE=92.9%  PMI=1.08
  Saved 20 routes -> dataoutput/Lidocaine_green_scores.json

[Paracetamol] Reading Paracetamol.txt ...
[1/3] Extracting SMILES and querying PubChem…
  Unique SMILES found: 77
  Sample (up to 10): ['[N-]=[N+]=Nc1ccc(O)cc1', 'CC(=O)Nc1ccc(F)cc1', 'CC(=O)Nc1ccc(OC(C)=O)cc1', 'CC(=O)CC(C)=O', 'O=[N+]([O-])c1cc(Br)c(O)c(Br)c1', 'CC(=O)Nc1cc(Br)c(O)c(Br)c1', 'O=[N+]([O-])c1ccc(OC2OC(CO)C(O)C(O)C2O)cc1', 'COc1ccc(CO)cc1', 'COc1ccc(CBr)cc1', 'CC(=O)Cl']


  PubChem lookup: 100%|██████████| 77/77 [00:34<00:00,  2.22it/s]


  Hazard entries retrieved: 48
[2/3] Calling LLM…
[3/3] Parsing, validating and ranking…

✅ Evaluated 103 routes.
   🥇 Best route: Graph 94  score=0.550  AE=91.5%  PMI=1.08
  Saved 103 routes -> dataoutput/Paracetamol_green_scores.json

[Phenacetin] Reading Phenacetin.txt ...
[1/3] Extracting SMILES and querying PubChem…
  Unique SMILES found: 80
  Sample (up to 10): ['CC(C)(C)OC(=O)CO', 'CCBr', 'CCOc1ccc(NC(=O)CO)cc1', 'O=CCI', 'CCOS(=O)(=O)OCC', 'CCCl', 'CC(=O)Cl', 'CCOc1ccc(NC(C)=O)cc1', 'C[O-]', 'C=CCI']


  PubChem lookup: 100%|██████████| 79/79 [00:33<00:00,  2.33it/s]


  Hazard entries retrieved: 52
[2/3] Calling LLM…
[3/3] Parsing, validating and ranking…

✅ Evaluated 96 routes.
   🥇 Best route: Graph 1  score=0.574  AE=98.3%  PMI=1.02
  Saved 96 routes -> dataoutput/Phenacetin_green_scores.json


BATCH EVALUATION SUMMARY
  Drug                 Status                       Routes  Best (graph / score)
------------------------------------------------------------
  Aspirin              skipped (cached)                200  
  Atorvastatin         done                              3  Graph 1 / 0.497
  Benzocaine           done                             88  Graph 39 / 0.580
  Celecoxib            done                             78  Graph 2 / 0.507
  Erlotinib            done                             90  Graph 2 / 0.594
  Favipiravir          done                            150  Graph 88 / 0.660
  Ibuprofen            done                            102  Graph 91 / 0.770
  Lidocaine            done                             20  Graph 2 / 0.466
  P